# Common Utilities and Configuration

This notebook contains shared utility functions and configurations for the SDV data generation pipeline.

In [ ]:
import pandas as pd
import numpy as np
import os
from pathlib import Path
import logging
import uuid
import math

## Setup Paths and Logging

In [ ]:
PROJECT_ROOT = Path('.').resolve().parent
WAREHOUSE_DIR = PROJECT_ROOT / 'warehouse'
SDV_OUT_DIR = PROJECT_ROOT / 'sdv-out'
EXCEL_OUT_DIR = SDV_OUT_DIR / 'excel'

# Create directories if they don't exist
for d in [WAREHOUSE_DIR, SDV_OUT_DIR, EXCEL_OUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

## Utility Functions

In [ ]:
def detect_excel_header(file_path):
    """
    Attempt to dynamically detect the true header row in a KiotViet Excel template.
    Usually it's somewhere in the first 10 rows.
    """
    try:
        df_sample = pd.read_excel(file_path, nrows=10, header=None)
        for i, row in df_sample.iterrows():
            # Detect a row that has many non-null string values, which suggests it's a header.
            if row.astype(str).str.len().sum() > 20:
                return i
        return 0
    except Exception as e:
        logger.error(f"Error reading {file_path}: {e}")
        return 0

def validate_uuid(val):
    try:
        uuid.UUID(str(val))
        return True
    except ValueError:
        return False

def export_chunks_to_excel(df, base_filename, out_dir, chunk_size=1000, column_mapping=None):
    """
    Exports a DataFrame to multiple Excel files if it exceeds chunk_size.
    Applies column mapping if provided.
    """
    if column_mapping:
        export_df = df.rename(columns=column_mapping)
    else:
        export_df = df.copy()
        
    total_rows = len(export_df)
    num_chunks = math.ceil(total_rows / chunk_size)
    
    for i in range(num_chunks):
        start_idx = i * chunk_size
        end_idx = min((i + 1) * chunk_size, total_rows)
        chunk = export_df.iloc[start_idx:end_idx]
        
        output_file = out_dir / f"{base_filename}_part_{i+1}.xlsx"
        chunk.to_excel(output_file, index=False)
        logger.info(f"Exported {len(chunk)} rows to {output_file}")

# Example Mappings (Update these according to actual KiotViet templates)
CUSTOMER_COL_MAP = {
    'customer_code': 'Mã khách hàng',
    'customer_name': 'Tên khách hàng',
    'phone': 'Điện thoại',
    'address': 'Địa chỉ'
}

PRODUCT_COL_MAP = {
    'product_code': 'Mã hàng',
    'product_name': 'Tên hàng',
    'category': 'Nhóm hàng',
    'cost_price': 'Giá vốn',
    'sale_price': 'Giá bán',
    'stock_on_hand': 'Tồn kho'
}

EMPLOYEE_COL_MAP = {
    'employee_code': 'Mã nhân viên',
    'employee_name': 'Tên nhân viên',
    'role': 'Vai trò',
    'branch': 'Chi nhánh'
}